# Clase 043 — SQL fundamental

**Parte 0** · Tanimura caps. 1-3.

> 🎯 SELECT/WHERE/JOIN/GROUP BY/HAVING + orden lógico de ejecución.

> ⏱️ ~120 min

## ⚙️ Setup — SQLite en memoria

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(':memory:')
con.executescript('''
CREATE TABLE clientes (
    cliente_id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    pais TEXT,
    plan TEXT
);
CREATE TABLE ordenes (
    orden_id INTEGER PRIMARY KEY,
    cliente_id INTEGER REFERENCES clientes(cliente_id),
    fecha DATE,
    monto REAL
);

INSERT INTO clientes (nombre, pais, plan) VALUES
    ('Ana',  'ES', 'pro'),
    ('Bob',  'ES', 'free'),
    ('Cris', 'CL', 'pro'),
    ('Dan',  'MX', 'free'),
    ('Eli',  'ES', 'pro');

INSERT INTO ordenes (cliente_id, fecha, monto) VALUES
    (1, '2024-01-15', 120),
    (1, '2024-02-20',  80),
    (1, '2024-03-12', 150),
    (1, '2024-04-05',  60),
    (2, '2024-02-10',  40),
    (3, '2024-01-22', 200),
    (3, '2024-03-15', 180),
    (5, '2024-02-28', 300),
    (5, '2024-03-22',  90);
''')
print('OK')

## 1️⃣ SELECT + WHERE

Operadores típicos: `=`, `<>` (distinto), `IN`, `BETWEEN`, `LIKE` (con wildcards `%` y `_`), `IS NULL`.

In [ ]:
q = '''
SELECT cliente_id, nombre, plan
FROM clientes
WHERE pais = 'ES' AND plan = 'pro'
'''
print(pd.read_sql(q, con))

## 2️⃣ JOIN — inner y left

In [ ]:
# INNER: solo clientes con orden
q = '''
SELECT c.nombre, o.fecha, o.monto
FROM clientes c
INNER JOIN ordenes o ON c.cliente_id = o.cliente_id
ORDER BY o.fecha
LIMIT 5
'''
print('INNER:')
print(pd.read_sql(q, con))

# LEFT: TODOS los clientes (incl. los sin orden)
q = '''
SELECT c.nombre, COUNT(o.orden_id) AS n_ordenes
FROM clientes c
LEFT JOIN ordenes o ON c.cliente_id = o.cliente_id
GROUP BY c.cliente_id, c.nombre
ORDER BY n_ordenes DESC
'''
print('\nLEFT + GROUP:')
print(pd.read_sql(q, con))

## 3️⃣ GROUP BY + agregadas

In [ ]:
q = '''
SELECT c.pais,
       COUNT(DISTINCT c.cliente_id) AS n_clientes,
       COUNT(o.orden_id)             AS n_ordenes,
       ROUND(SUM(o.monto), 2)         AS total_monto,
       ROUND(AVG(o.monto), 2)         AS monto_medio
FROM clientes c
LEFT JOIN ordenes o ON c.cliente_id = o.cliente_id
GROUP BY c.pais
ORDER BY total_monto DESC
'''
print(pd.read_sql(q, con))

## 4️⃣ HAVING — filtrar agregados

**WHERE** filtra **antes** de agrupar (rows individuales).  
**HAVING** filtra **después** de agrupar (grupos).

In [ ]:
q = '''
SELECT c.cliente_id, c.nombre,
       COUNT(o.orden_id) AS n_ord,
       SUM(o.monto)      AS total
FROM clientes c
INNER JOIN ordenes o ON c.cliente_id = o.cliente_id
GROUP BY c.cliente_id, c.nombre
HAVING COUNT(o.orden_id) >= 2 AND SUM(o.monto) > 200
ORDER BY total DESC
'''
print(pd.read_sql(q, con))

## 5️⃣ Orden lógico de ejecución

Escribimos:
```sql
SELECT col, AGG(otra)
FROM tabla
WHERE cond
GROUP BY col
HAVING AGG(otra) > 100
ORDER BY col
LIMIT 10
```

Pero SQL **ejecuta en este orden** (es lo que importa para entender errores):

```
1. FROM       ← lee las tablas
2. WHERE      ← filtra filas individuales
3. GROUP BY   ← agrupa
4. HAVING     ← filtra grupos
5. SELECT     ← calcula expresiones del select
6. ORDER BY   ← ordena
7. LIMIT      ← corta
```

Por eso `WHERE SUM(...)` da error: WHERE corre **antes** de GROUP BY, no hay agregado todavía. Usa HAVING.

## 6️⃣ DuckDB — drop-in con superpoderes

```python
import duckdb
con = duckdb.connect(':memory:')
con.execute("CREATE TABLE clientes AS SELECT * FROM 'clientes.csv'")
# Soporta SQL estándar moderno + window functions + lee CSV/Parquet directo
```

DuckDB es como SQLite pero pensado para análisis (columnar, vectorizado). Lo verás en clase 043.

## ✅ Checklist

- [ ] Escribo SELECT con WHERE, operadores variados
- [ ] Hago INNER y LEFT JOIN según el caso
- [ ] Uso GROUP BY + agregadas (COUNT/SUM/AVG)
- [ ] Sé cuándo HAVING (no WHERE) sobre agregados
- [ ] Recito el orden lógico FROM→WHERE→GROUP→HAVING→SELECT→ORDER→LIMIT

## 📝 Homework

Ver `README.md`. SQLite en memoria, 5 consultas progresivas, mismo en DuckDB.

## 📖 Definiciones y características

**SQL (Structured Query Language)**

Lenguaje declarativo para bases de datos relacionales. Describes **qué** quieres (no cómo) y el motor lo ejecuta. Estandarizado pero con dialectos (SQLite, PostgreSQL, MySQL, BigQuery).

**Orden lógico vs escrito**

**Escribes**: SELECT-FROM-WHERE-GROUP-HAVING-ORDER. **Ejecuta**: FROM-WHERE-GROUP-HAVING-SELECT-ORDER-LIMIT. Por eso `WHERE SUM(...)` falla (aún no agrupado) — usa HAVING.

**JOIN**

Combina filas de 2+ tablas por una key. Tipos: INNER (intersección), LEFT (todo left + match right), RIGHT (espejo), FULL OUTER (todo unión), CROSS (producto cartesiano).

**GROUP BY + HAVING**

**GROUP BY**: agrupa filas por valor(es). **HAVING**: filtra los **grupos** después de agregar (no se puede con WHERE).

**Funciones de agregación**

Operan sobre grupos: `COUNT(*)`, `COUNT(DISTINCT x)`, `SUM`, `AVG`, `MIN`, `MAX`, `STDDEV`. Devuelven UN valor por grupo.

**DuckDB**

Motor SQL embebido (como SQLite) pero **columnar** y optimizado para analytics. Lee CSV/Parquet directo (`FROM 'file.csv'`). Drop-in para queries analíticas, mucho más rápido que SQLite en agregados.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `column "x" must appear in GROUP BY clause or be used in an aggregate function` | Seleccionaste col no agregada ni en GROUP BY. **Fix**: añade a GROUP BY, o agrega con `MAX(x)`, `MIN(x)` (cuando da igual). |
| `WHERE SUM(monto) > 100` falla | WHERE corre ANTES de GROUP. **Fix**: usa `HAVING SUM(monto) > 100`. |
| INNER JOIN pierde filas que esperaba ver | La key no matchea (NULL, tipos, espacios). **Fix**: LEFT JOIN + `WHERE r.id IS NULL` para diagnóstico. |
| `COUNT(col)` devuelve menos que `COUNT(*)` | `COUNT(col)` ignora NULL en esa columna. **Fix**: si quieres todas las filas, `COUNT(*)`. |
| `SELECT *` después de JOIN trae cols duplicadas con mismo nombre | Ambas tablas tienen `id`. **Fix**: aliasea: `SELECT c.id AS cliente_id, o.id AS orden_id`. |

## ❓ Preguntas frecuentes

**❓ ¿`COUNT(*)` o `COUNT(1)`?**

**Equivalentes** en motores modernos (parser optimiza). `COUNT(*)` es más legible — úsalo.

**❓ ¿Cuándo `DISTINCT`?**

Cuando hay filas duplicadas que no deberían contarse. **Cuidado**: en SELECT con muchas cols puede ser caro. Mejor agrupa con GROUP BY si vas a agregar después.

**❓ ¿`UNION` o `UNION ALL`?**

**`UNION`** quita duplicados (más caro). **`UNION ALL`** mantiene todo (más rápido). Usa ALL si sabes que no hay duplicados (más común).

**❓ ¿SQLite o PostgreSQL para aprender?**

**SQLite** para arrancar (sin servidor, 1 archivo). El SQL es 90% igual. Migras a PostgreSQL cuando necesites: tipos avanzados, concurrencia, escala, JSON nativo.

**❓ ¿Cómo trato fechas en SQL?**

Cada motor su dialecto. SQLite: strings ISO `'2024-01-15'` + `date()`, `strftime()`. PostgreSQL: tipo `DATE`/`TIMESTAMP` nativo. Estándar: `DATE '2024-01-15'`.

## 🔗 Referencias

- Tanimura, *SQL for Data Scientists*
- [SQLite SELECT](https://www.sqlite.org/lang_select.html)
- [DuckDB](https://duckdb.org/docs/)

➡️ **Siguiente:** [044 — SQL avanzado](../044-sql-avanzado-ctes-window-functions-subqueries-correlacionadas/README.md)